In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline
import matplotlib.animation as animation
from ipywidgets import interact, FloatSlider, widgets
from IPython.display import display

# ==========================================
# SIMULATION CONFIGURATION (Global Variables)
# ==========================================
sa = 30           # Steering angle limit in degrees
wb = 2            # Wheelbase in meters
k_gain = 0.5      # Default position gain
ks_gain = 1.0     # Default softening gain
v_target = 10.0   # Target speed (m/s)
lat_offset = 20.0 # Initial lateral offset from path (meters)
init_yaw_deg = 45.0 # Initial vehicle heading in DEGREES (0 = East, 90 = North)
time_limit = 20.0 # Simulation run time (seconds)
dt = 0.05         # Physics time step (seconds)
gains_to_test = [0.1, 0.5, 1.5, 2.0]  # Different k gains for experimentation

# ==========================================
# VEHICLE & CONTROLLER CLASSES
# ==========================================
class KinematicBicycleModel:
    def __init__(self, x=0.0, y=0.0, yaw=0.0, v=0.0, L=2.0): 
        self.x = x
        self.y = y
        self.yaw = yaw
        self.v = v
        self.L = L  

    def update(self, a, delta, dt, max_steer_deg): 
        delta = np.clip(delta, -np.radians(max_steer_deg), np.radians(max_steer_deg))
        
        self.x += self.v * np.cos(self.yaw) * dt
        self.y += self.v * np.sin(self.yaw) * dt
        self.yaw += (self.v / self.L) * np.tan(delta) * dt
        self.v += a * dt

class StanleyController:
    def __init__(self, k=0.5, ks=1.0): 
        self.k = k
        self.ks = ks

    def compute_steering(self, vehicle, path_x, path_y, path_yaw):
        # 1. Find the closest point on the path to the FRONT axle
        fx = vehicle.x + vehicle.L * np.cos(vehicle.yaw)
        fy = vehicle.y + vehicle.L * np.sin(vehicle.yaw)
        
        dx = [fx - x for x in path_x]
        dy = [fy - y for y in path_y]
        distances = np.hypot(dx, dy)
        target_idx = np.argmin(distances)

        # 2. Correct Cross-Track Error calculation (Using 2D Cross Product)
        dx_front = fx - path_x[target_idx]
        dy_front = fy - path_y[target_idx]
        
        path_vec_x = np.cos(path_yaw[target_idx])
        path_vec_y = np.sin(path_yaw[target_idx])
        
        error_front_axle = dx_front * path_vec_y - dy_front * path_vec_x

        # 3. Correct Heading Error (Path Yaw minus Vehicle Yaw)
        yaw_error = path_yaw[target_idx] - vehicle.yaw
        yaw_error = np.arctan2(np.sin(yaw_error), np.cos(yaw_error))

        # 4. Stanley Control Law
        crosstrack_steering = np.arctan2(self.k * error_front_axle, vehicle.v + self.ks)
        delta = yaw_error + crosstrack_steering
        
        return delta, error_front_axle

# ==========================================
# SIMULATION CORE ENGINE
# ==========================================
def run_simulation(path_x, path_y, path_yaw, k_param, ks_param):
    vehicle = KinematicBicycleModel(x=path_x[0], y=path_y[0] - lat_offset, yaw=np.radians(init_yaw_deg), v=2.0, L=wb)
    controller = StanleyController(k=k_param, ks=ks_param)
    
    time = 0.0
    logs = []

    while time < time_limit:
        delta, cte = controller.compute_steering(vehicle, path_x, path_y, path_yaw)
        dynamic_v_target = v_target * np.cos(delta) 
        tire_scrub_penalty = 1.2 * np.sin(abs(delta)) * vehicle.v
        a = 0.5 * (dynamic_v_target - vehicle.v) - tire_scrub_penalty
        
        logs.append({
            'time': time, 'x': vehicle.x, 'y': vehicle.y, 
            'yaw': vehicle.yaw, 'v': vehicle.v, 'cte': cte, 'delta': delta
        })
        
        vehicle.update(a, delta, dt, max_steer_deg=sa)
        time += dt
        
    return pd.DataFrame(logs)

# ==========================================
# EXECUTION & EXPERIMENTATION
# ==========================================

# 1. Define raw track waypoints and build cubic spline
waypoints_x = [0.0, 20.0,  40.0, 60.0,  80.0, 100.0, 110.0, 100.0, 80.0]
waypoints_y = [0.0,  5.0, -10.0,  0.0,  15.0,  10.0,  -5.0, -20.0, -25.0]

t_waypoints = np.arange(len(waypoints_x))
spline_x = CubicSpline(t_waypoints, waypoints_x, bc_type='natural')
spline_y = CubicSpline(t_waypoints, waypoints_y, bc_type='natural')

t_fine = np.linspace(0, len(waypoints_x) - 1, 1000)
path_x = spline_x(t_fine)
path_y = spline_y(t_fine)

dx_dt = spline_x(t_fine, 1)
dy_dt = spline_y(t_fine, 1)
path_yaw = np.arctan2(dy_dt, dx_dt)

# 2. Execute batch runs
results = {}
for g in gains_to_test:
    run_id = f"Gain_k_{g}"
    results[run_id] = run_simulation(path_x, path_y, path_yaw, k_param=g, ks_param=ks_gain)

# 3. Create the Interactive Plotting Function
# 3. Create the Interactive Plotting Function
def update_dashboard(target_time):
    # Setup a 2x3 layout matrix
    fig = plt.figure(figsize=(18, 11))
    
    # --- Subplot 1: Visual Map Tracker (Top Left) ---
    ax_map = fig.add_subplot(2, 3, 1)
    ax_map.plot(path_x, path_y, 'r--', linewidth=2, label='Smoothed Reference Path')
    ax_map.plot(waypoints_x, waypoints_y, 'ko', markersize=6, label='Raw Waypoints')
    
    # Dictionary to hold line colors so we can reuse them across all subplots
    run_colors = {}
    
    for run_id, df in results.items():
        # Store the color assigned to this specific simulation run
        line, = ax_map.plot(df['x'], df['y'], label=f"{run_id}", alpha=0.6)
        run_colors[run_id] = line.get_color()
        
        closest_idx = (df['time'] - target_time).abs().argmin()
        current_row = df.iloc[closest_idx]
        ax_map.plot(current_row['x'], current_row['y'], marker='o', markersize=10, 
                    color=run_colors[run_id], markeredgecolor='black', markeredgewidth=1.5)
        
    ax_map.set_title(f'Vehicle Positions at t = {target_time:.2f}s')
    ax_map.set_xlabel('X Position (m)')
    ax_map.set_ylabel('Y Position (m)')
    ax_map.legend()
    ax_map.grid(True)
    ax_map.axis('equal') 

    # --- Subplot 2: Cross-Track Error Timeline (Top Center) ---
    ax_cte = fig.add_subplot(2, 3, 2)
    for run_id, df in results.items():
        ax_cte.plot(df['time'], df['cte'], color=run_colors[run_id], label=f"{run_id}")
    ax_cte.axvline(x=target_time, color='k', linestyle='--', linewidth=1.5)
    ax_cte.set_xlabel('Time (s)')
    ax_cte.set_ylabel('Cross-Track Error (m)')
    ax_cte.set_title('Cross-Track Error Timeline')
    ax_cte.grid(True)

    # --- Subplot 3: Circular Heading Compass Plot (Top Right) ---
    ax_compass = fig.add_subplot(2, 3, 3, projection='polar')
    
    # Configure grid orientation: East = 0 deg (Right), North = 90 deg (Up)
    ax_compass.set_theta_zero_location("E") 
    ax_compass.set_theta_direction(1) # 1 = Counter-Clockwise
    ax_compass.set_yticklabels([])     # Hide the radial coordinate circles
    
    for run_id, df in results.items():
        closest_idx = (df['time'] - target_time).abs().argmin()
        current_yaw = df.iloc[closest_idx]['yaw']
        
        # FIXED: Forces the radial line and marker dot to share the exact color from the map
        ax_compass.plot([0, current_yaw], [0, 1.0], color=run_colors[run_id], linewidth=2.5)
        ax_compass.plot(current_yaw, 1.0, marker='o', markersize=10, color=run_colors[run_id],
                        label=f"{run_id}", markeredgecolor='black', markeredgewidth=1.5)

    ax_compass.set_title("Current Vehicle Heading ($\psi$)", pad=20)
    # Deduplicate legend entries to keep it concise and clean
    handles, labels = ax_compass.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax_compass.legend(by_label.values(), by_label.keys(), loc='lower center', bbox_to_anchor=(0.5, -0.3), ncol=2)

    # --- Subplot 4: Steering Angle Timeline (Bottom Left) ---
    ax_steer = fig.add_subplot(2, 3, 4)
    for run_id, df in results.items():
        steering_deg = np.degrees(df['delta'])
        ax_steer.plot(df['time'], steering_deg, color=run_colors[run_id], label=f"{run_id}")
    ax_steer.axhline(y=sa, color='grey', linestyle=':', label='Max Mech Limit')
    ax_steer.axhline(y=-sa, color='grey', linestyle=':')
    ax_steer.axvline(x=target_time, color='k', linestyle='--', linewidth=1.5)
    ax_steer.set_xlabel('Time (s)')
    ax_steer.set_ylabel('Steering Angle (Degrees)')
    ax_steer.set_title('Steering Input Timeline')
    ax_steer.grid(True)

    # --- Subplot 5: Velocity Tracking Timeline (Bottom Center) ---
    ax_vel = fig.add_subplot(2, 3, 5)
    ax_vel.axhline(y=v_target, color='r', linestyle='--', linewidth=2, label='Target Speed')
    for run_id, df in results.items():
        ax_vel.plot(df['time'], df['v'], color=run_colors[run_id], label=f"{run_id}")
    ax_vel.axvline(x=target_time, color='k', linestyle='--', linewidth=1.5)
    ax_vel.set_xlabel('Time (s)')
    ax_vel.set_ylabel('Velocity (m/s)')
    ax_vel.set_title('Vehicle Speed Timeline')
    ax_vel.grid(True)

    # --- Subplot 6: Blank Layout Spacer (Bottom Right) ---
    ax_blank = fig.add_subplot(2, 3, 6)
    ax_blank.axis('off')

    plt.tight_layout()
    plt.show()

# 4. Initialize and display the interactive controller widget
interact(
    update_dashboard, 
    target_time=FloatSlider(
        value=0.0, 
        min=0.0, 
        max=time_limit, 
        step=dt, 
        description='Sim Time (s):',
        continuous_update=True,
        layout=widgets.Layout(width='60%')
    )
);

interactive(children=(FloatSlider(value=0.0, description='Sim Time (s):', layout=Layout(width='60%'), max=20.0…